## Library loading

In [1]:
import pandas as pd
import geopandas as gpd

## Year lookups

In [2]:
fy_lookup = pd.read_csv("../../data/Financial year lookup.csv")

fy_lookup['year_month'] = pd.to_datetime(fy_lookup['year_month'], format='%d/%m/%Y')

In [3]:
fy_lookup

,Financial year,mid year pop estimate month,annual year,Quarters,dwelling_stock_year,year_month
0,1967/68,1967,1967,1967 Q3,1967,1967-07-01
1,1967/68,1967,1967,1967 Q3,1967,1967-08-01
2,1967/68,1967,1967,1967 Q3,1967,1967-09-01
3,1967/68,1967,1967,1967 Q4,1967,1967-10-01
4,1967/68,1967,1967,1967 Q4,1967,1967-11-01
...,...,...,...,...,...,...
709,2026/27,2026,2026,2026 Q3,2026,2026-08-01
710,2026/27,2026,2026,2026 Q3,2026,2026-09-01
711,2026/27,2026,2026,2026 Q4,2026,2026-10-01
712,2026/27,2026,2026,2026 Q4,2026,2026-11-01


## Shapefile

In [4]:
# --- Parameters ---
shapefile_path = "../../data/Local_Authority_Districts_(May_2025)_Boundaries_UK_BFC_(V2)/Local_Authority_Districts_(May_2025)_Boundaries_UK_BFC_(V2).shp"  # Change to your shapefile path
id_column = "LAD25CD"       # Column where first letter is E/W/S/N
name_column = "LAD25NM"   # Column with local authority names

# --- Load shapefile ---
gdf = gpd.read_file(shapefile_path)

# Filter to only England
gdf = gdf[gdf[id_column].str[0].isin(["E"])]

# Ensure consistent projection
gdf = gdf.to_crs(epsg=27700)  # British National Grid

In [5]:
gdf

,FID,LAD25CD,LAD25NM,LAD25NMW,BNG_E,BNG_N,LONG,LAT,Shape__Are,Shape__Len,GlobalID,geometry
0,1,E06000001,Hartlepool,None,447161,531473,-1.270174,54.676132,9.388700e+07,73982.408648,847f2c4b-a7cf-4c71-840c-0864853246d5,"MULTIPOLYGON (((450154.6 525938.201, 450140.09..."
1,2,E06000002,Middlesbrough,None,451141,516887,-1.210998,54.544679,5.388156e+07,44481.691242,f1925b75-6267-417d-a77a-05cdc4c6b1b3,"MULTIPOLYGON (((446854.7 517192.7, 446854.281 ..."
2,3,E06000003,Redcar and Cleveland,None,464330,519596,-1.006565,54.567520,2.451071e+08,97674.177085,36b1db27-3dfa-4ed6-8e81-36bf3abeeacc,"MULTIPOLYGON (((451747.397 520561.1, 451792.20..."
3,4,E06000004,Stockton-on-Tees,None,444940,518179,-1.306646,54.556876,2.049433e+08,123628.292301,22a6adf7-e812-4c09-89b1-6753ec35de93,"MULTIPOLYGON (((447177.704 517811.797, 447176...."
4,5,E06000005,Darlington,None,428029,515648,-1.568356,54.535345,1.974895e+08,107285.516031,309364b5-9b5c-4f9c-81f5-8a3a79699fd1,"POLYGON ((423496.602 524724.299, 423497.204 52..."
...,...,...,...,...,...,...,...,...,...,...,...,...
291,292,E09000029,Sutton,None,527357,163639,-0.172265,51.357555,4.384639e+07,39931.559198,becd0e65-98c8-4e28-b3ed-d13f60c1d51a,"POLYGON ((527085.942 167617.691, 527087.728 16..."
292,293,E09000030,Tower Hamlets,None,536340,181452,-0.036476,51.515548,1.977087e+07,28755.059699,56e55229-e9a1-4dfc-b898-e76407182c4b,"POLYGON ((536776.386 184446.881, 536777.6 1844..."
293,294,E09000031,Waltham Forest,None,537328,190278,-0.018803,51.594619,3.881023e+07,33346.583215,4bc26b95-0c65-4798-929c-8c4a89bece89,"POLYGON ((537921.098 196048.201, 537947.499 19..."
294,295,E09000032,Wandsworth,None,525152,174138,-0.200217,51.452402,3.426187e+07,38840.162299,25eb5509-ec07-424d-b406-c1c28a2dce5c,"POLYGON ((530059.62 177871.478, 530066.94 1778..."


## Function to update LA codes and names - default for dwelling stock

In [6]:
## Function to replace LA code and name pair to fix LA changes
def replace_la_pair(df, old_code, old_area, new_code, new_area, code_col='NewONScode', area_col='Area'):
    """Replace NewONScode and Area where both match old values.
    Prints number of rows changed and returns the dataframe."""
    mask = (df[code_col] == old_code) & (df[area_col] == old_area)
    changed = int(mask.sum())
    if changed:
        df.loc[mask, [code_col, area_col]] = [new_code, new_area]
    print(f"Updated rows ({old_code}, {old_area} -> {new_code}, {new_area}): {changed}")
    return df

## Population data loading

In [7]:
pop_by_la = pd.read_excel('../../data/population estimates.xlsx', sheet_name='Data', skiprows=6, na_values=['-'])

# detect columns that look like year (e.g."1997")
year_cols = pop_by_la.filter(regex=r'(^\d{4}$)').columns

id_vars = [c for c in pop_by_la.columns if c not in year_cols]

pop_by_la = pd.melt(
    pop_by_la,
    id_vars=id_vars,
    value_vars=year_cols,
    var_name='year',
    value_name='population'
)

# year to int
pop_by_la['year'] = pop_by_la['year'].astype(int)   
              
# filter to keep only England LAs
pop_by_la = pop_by_la[pop_by_la['LAD25CD'].str[0].isin(["E"])]

pop_by_la = pd.merge(
    pop_by_la,
    fy_lookup[['year_month','mid year pop estimate month']],
    how='left',
    left_on='year',
    right_on='mid year pop estimate month'
)


## Annual survey of earnings and hours

In [8]:
ashe_la = pd.read_excel('../../data/earnings.xlsx', sheet_name='Median earnings by LA', na_values=['-'])

# detect columns that look like year (e.g."1997")
year_cols = ashe_la.filter(regex=r'(^\d{4}$)').columns

id_vars = [c for c in ashe_la.columns if c not in year_cols]

ashe_la = pd.melt(
    ashe_la,
    id_vars=id_vars,
    value_vars=year_cols,
    var_name='year',
    value_name='ashe_weekly'
)

# year to int
ashe_la['year'] = ashe_la['year'].astype(int)   
ashe_la['financial_year'] = ashe_la['year'].astype(int).apply(lambda y: f"{y-1}/{str(y)[-2:]}")

ashe_la.drop(columns=['year'], inplace=True)
              
ashe_la = pd.merge(
    ashe_la,
    fy_lookup[['year_month','Financial year']],
    how='left',
    left_on='financial_year',
    right_on='Financial year'
)

ashe_la.drop(columns=['financial_year','Financial year'], inplace=True)

ashe_la.rename(columns={'Code': 'LAD25CD', 'Description': 'Description_ashe'}, inplace=True)

In [9]:
ashe_la

,Description_ashe,LAD25CD,ashe_weekly,year_month
0,Adur,E07000223,327.2,2007-04-01
1,Adur,E07000223,327.2,2007-05-01
2,Adur,E07000223,327.2,2007-06-01
3,Adur,E07000223,327.2,2007-07-01
4,Adur,E07000223,327.2,2007-08-01
...,...,...,...,...
63931,York UA,E06000014,619.2,2024-11-01
63932,York UA,E06000014,619.2,2024-12-01
63933,York UA,E06000014,619.2,2025-01-01
63934,York UA,E06000014,619.2,2025-02-01


## Base rate

In [10]:
base_rate = pd.read_excel('../../data/base rate.xlsx', sheet_name='Sheet1')

## Claimant count

In [11]:
claimant_count = pd.read_excel('../../data/claimant count.xlsx', sheet_name='claimant count (proportion)', skiprows=7, skipfooter=40)  

year_cols = claimant_count.filter(regex=r'(^\d{4}-\d{2}-\d{2} 00:00:00$)').columns

id_vars = [c for c in claimant_count.columns if c not in year_cols]

claimant_count = pd.melt(
    claimant_count,
    id_vars=id_vars,
    value_vars=year_cols,
    var_name='year_month',
    value_name='claimant_count_prop'
)
              
# filter to keep only England LAs
claimant_count = claimant_count[claimant_count['mnemonic'].str[0].isin(["E"])]

claimant_count['year_month'] = pd.to_datetime(claimant_count['year_month']).dt.date
claimant_count['year_month'] = pd.to_datetime(claimant_count['year_month'], format='%Y-%m-%d')

In [12]:
claimant_count.info()

<class 'pandas.core.frame.DataFrame'>
Index: 119584 entries, 0 to 146181
Data columns (total 4 columns):
 #   Column                                                  Non-Null Count   Dtype         
---  ------                                                  --------------   -----         
 0   local authority: district / unitary (as of April 2023)  119584 non-null  object        
 1   mnemonic                                                119584 non-null  object        
 2   year_month                                              119584 non-null  datetime64[ns]
 3   claimant_count_prop                                     119584 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 4.6+ MB


## Planning applications

In [13]:
planning_apps_granted_prop = pd.read_csv('../../data/planning data (1).csv', skiprows=2, na_values=['..'], usecols=range(6))

planning_apps_granted_prop['Prop_granted'] = planning_apps_granted_prop['Total granted; grand total (all)']/planning_apps_granted_prop['Total decisions; grand total (all)']

planning_apps_granted_prop = pd.merge(
    planning_apps_granted_prop,
    fy_lookup[['year_month','Quarters']],
    how='left',
    left_on='Quarter',
    right_on='Quarters'
).drop(
     columns=['Region', 'Quarter', 'Quarters', 'Total granted; grand total (all)', 'Total decisions; grand total (all)']
 ).rename(columns={'LPACD': 'LAD25CD', 'LPANM': 'LAD25NM'})

In [14]:
planning_apps_granted_prop

,LAD25NM,LAD25CD,Prop_granted,year_month
0,Adur,E07000223,0.875912,2001-01-01
1,Adur,E07000223,0.875912,2001-02-01
2,Adur,E07000223,0.875912,2001-03-01
3,Allerdale,E07000026,0.916667,2001-01-01
4,Allerdale,E07000026,0.916667,2001-02-01
...,...,...,...,...
102652,York,E06000014,0.804813,2025-05-01
102653,York,E06000014,0.804813,2025-06-01
102654,Yorkshire Dales National Park,E26000012,0.947368,2025-04-01
102655,Yorkshire Dales National Park,E26000012,0.947368,2025-05-01


## Dwelling Stock

In [15]:
## Function to replace LA code and name pair to fix LA changes
def replace_la_pair(df, old_code, old_area, new_code, new_area, code_col='NewONScode', area_col='Area'):
    """Replace NewONScode and Area where both match old values.
    Prints number of rows changed and returns the dataframe."""
    mask = (df[code_col] == old_code) & (df[area_col] == old_area)
    changed = int(mask.sum())
    if changed:
        df.loc[mask, [code_col, area_col]] = [new_code, new_area]
    print(f"Updated rows ({old_code}, {old_area} -> {new_code}, {new_area}): {changed}")
    return df

In [16]:
dwelling_stock = pd.read_excel('../../data/dwelling stock.xlsx', sheet_name='LT_125_unrounded', skiprows=5, na_values=['[x]', '[z]'])

dwelling_stock = dwelling_stock.drop(columns=['OldONScode'])

# filter to keep only England LAs
dwelling_stock = dwelling_stock[dwelling_stock['NewONScode'].str[0].isin(["E"])]

year_cols = dwelling_stock.filter(regex=r'(^\d{4}$)').columns

id_vars = [c for c in dwelling_stock.columns if c not in year_cols]

dwelling_stock = pd.melt(
    dwelling_stock,
    id_vars=id_vars,
    value_vars=year_cols,
    var_name='year',
    value_name='dwelling_stock'
)

## Durham to County Durham UA 09 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E10000010', 'Durham', 'E06000047', 'County Durham UA')

# Cheshire East update 09 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000014', 'Congleton', 'E06000049', 'Cheshire East UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000015', 'Crewe and Nantwich', 'E06000049', 'Cheshire East UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000017', 'Macclesfield', 'E06000049', 'Cheshire East UA')

# Cheshire West update	09 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000013', 'Chester', 'E06000050', 'Cheshire West and Chester UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000016', 'Ellesmere Port and Neston', 'E06000050', 'Cheshire West and Chester UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000018', 'Vale Royal', 'E06000050', 'Cheshire West and Chester UA')

# Shropshire UA update 09 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E10000026', 'Shropshire', 'E06000051', 'Shropshire UA')

# Cornwall UA update 09 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000019', 'Caradon', 'E06000052', 'Cornwall UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000020', 'Carrick', 'E06000052', 'Cornwall UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000021', 'Kerrier', 'E06000052', 'Cornwall UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000022', 'North Cornwall', 'E06000052', 'Cornwall UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000023', 'Penwith', 'E06000052', 'Cornwall UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000024', 'Restormel', 'E06000052', 'Cornwall UA')

# Wiltshire UA update 09 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E10000033', 'Wiltshire', 'E06000054', 'Wiltshire UA')

# Bedford UA update 09 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000002', 'Bedford', 'E06000055', 'Bedford UA')

# Central Bedfordshire UA update 09 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000001', 'Mid Bedfordshire', 'E06000056', 'Central Bedfordshire UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000003', 'South Bedfordshire', 'E06000056', 'Central Bedfordshire UA')

# Northumberland UA update 09 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E10000022', 'Northumberland', 'E06000057', 'Northumberland UA')
	
# Bournemouth, christchurch and poole UA update '19 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E06000028', 'Bournemouth UA', 'E06000058', 'Bournemouth, Christchurch and Poole UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E06000029', 'Poole UA', 'E06000058', 'Bournemouth, Christchurch and Poole UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000048', 'Christchurch', 'E06000058', 'Bournemouth, Christchurch and Poole UA')

# Dorset UA update '19 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000049', 'East Dorset', 'E06000059', 'Dorset UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000050', 'North Dorset', 'E06000059', 'Dorset UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000051', 'Purbeck', 'E06000059', 'Dorset UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000052', 'West Dorset', 'E06000059', 'Dorset UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000053', 'Weymouth and Portland', 'E06000059', 'Dorset UA')

# Buckinghamshire UA update '19 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E10000002', 'Buckinghamshire', 'E06000060', 'Buckinghamshire UA')

# East Suffolk update '19 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000205', 'Suffolk Coastal', 'E07000244', 'East Suffolk')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000206', 'Waveney', 'E07000244', 'East Suffolk')

# West Suffolk update '19 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000201', 'Forest Heath', 'E07000245', 'West Suffolk')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000204', 'St. Edmundsbury', 'E07000245', 'West Suffolk')

# North Northamptonshire update '21 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000150', 'Corby', 'E06000061', 'North Northamptonshire')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000152', 'East Northamptonshire', 'E06000061', 'North Northamptonshire')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000153', 'Kettering', 'E06000061', 'North Northamptonshire')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000156', 'Wellingborough', 'E06000061', 'North Northamptonshire')

# West Northamptonshire update '21 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000151', 'Daventry', 'E06000062', 'West Northamptonshire')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000154', 'Northampton', 'E06000062', 'West Northamptonshire')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000155', 'South Northamptonshire', 'E06000062', 'West Northamptonshire')

# Cumberland update '23 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000026', 'Allerdale', 'E06000063', 'Cumberland')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000028', 'Carlisle', 'E06000063', 'Cumberland')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000029', 'Copeland', 'E06000063', 'Cumberland')

# Westmorland and Furness update '23 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000027', 'Barrow-in-Furness', 'E06000064', 'Westmorland and Furness')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000030', 'Eden', 'E06000064', 'Westmorland and Furness')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000031', 'South Lakeland', 'E06000064', 'Westmorland and Furness')

# North Yorkshire update '23 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E10000023', 'North Yorkshire', 'E06000065', 'North Yorkshire')

# Somerset update '23 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E10000027', 'Somerset', 'E06000066', 'Somerset')

# Sheffield and Barnsley ons recode in dwelling stock
dwelling_stock = replace_la_pair(dwelling_stock, 'E08000016', 'Barnsley', 'E08000038', 'Barnsley')
dwelling_stock = replace_la_pair(dwelling_stock, 'E08000019', 'Sheffield', 'E08000039', 'Sheffield')


# year to int
dwelling_stock['year'] = dwelling_stock['year'].astype(int)   

# Summarise dwelling_stock so each (NewONScode, Area, year) is unique
dwelling_stock = dwelling_stock.groupby(
    ['NewONScode', 'Area', 'year'],
    as_index=False
).agg({'dwelling_stock': 'sum'})  # use 'mean' or other agg if appropriate

dwelling_stock = pd.merge(
    dwelling_stock,
    fy_lookup[['year_month','dwelling_stock_year']],
    how='left',
    left_on='year',
    right_on='dwelling_stock_year'
).drop(columns = ['dwelling_stock_year'])



Updated rows (E10000010, Durham -> E06000047, County Durham UA): 24
Updated rows (E07000014, Congleton -> E06000049, Cheshire East UA): 24
Updated rows (E07000015, Crewe and Nantwich -> E06000049, Cheshire East UA): 24
Updated rows (E07000017, Macclesfield -> E06000049, Cheshire East UA): 24
Updated rows (E07000013, Chester -> E06000050, Cheshire West and Chester UA): 24
Updated rows (E07000016, Ellesmere Port and Neston -> E06000050, Cheshire West and Chester UA): 24
Updated rows (E07000018, Vale Royal -> E06000050, Cheshire West and Chester UA): 24
Updated rows (E10000026, Shropshire -> E06000051, Shropshire UA): 24
Updated rows (E07000019, Caradon -> E06000052, Cornwall UA): 24
Updated rows (E07000020, Carrick -> E06000052, Cornwall UA): 24
Updated rows (E07000021, Kerrier -> E06000052, Cornwall UA): 24
Updated rows (E07000022, North Cornwall -> E06000052, Cornwall UA): 24
Updated rows (E07000023, Penwith -> E06000052, Cornwall UA): 24
Updated rows (E07000024, Restormel -> E06000052

## ORR Station entry and exit numbers  by station and LA

2003/04 data had to be interpolated. This was done by taking the midpoint between 2002/03 and 2004/05. This was done directly in the Excel file.

I've also had to add the ONS LA codes for ease when joining the data. This was also done directly in the Excel file.

In [17]:
raw_orr_data = pd.read_excel("../../data/station entries and exits.xlsx", sheet_name="1415a_Entries_and_Exits", skiprows = 3, na_values = ['[z]', '[x]'])

In [18]:
raw_orr_data = raw_orr_data[~raw_orr_data['Region'].isin(["Scotland", "[z]"])& ~raw_orr_data['Region'].isna()]



In [19]:
raw_orr_data = pd.melt(
    raw_orr_data,
    id_vars=[col for col in raw_orr_data.columns if not pd.Series(col).str.match(r'^\d{4}/\d{2}$')[0]],   # columns to keep fixed
    value_vars=raw_orr_data.filter(regex=r'^\d{4}/\d{2}$').columns,    # columns to unpivot (optional)
    var_name='Financial year',                    # name for new variable column
    value_name='Value'                      # name for new value column
)

In [20]:
orr_data = raw_orr_data.groupby(
    ['Financial year', 'Local authority: district or unitary', 'Local authority code'],
      as_index=False
)['Value'].sum()

In [21]:
orr_data = pd.merge(
    orr_data,
    fy_lookup[['year_month','Financial year']],
    how='left',
    left_on='Financial year',
    right_on='Financial year'
)

In [22]:
orr_data.rename(columns={'Local authority code': 'LAD25CD', 'Local authority: district or unitary': 'LAD25NM'}, inplace=True)

## GDP data

In [23]:
gdp_data = pd.read_excel("../../data/monthlygdpto4dp.xlsx", sheet_name="Data_table", skiprows=3)

In [24]:
gdp_data['Month'] = pd.to_datetime(gdp_data['Month'], format='%Y%b')

In [25]:
gdp_data = gdp_data[['Month', 'Monthly GDP (A-T)']]
gdp_data.rename(columns={'Monthly GDP (A-T)': 'GDP'}, inplace=True)

In [26]:
gdp_orr_data = pd.merge(orr_data, gdp_data, how='left', left_on='year_month', right_on='Month')

In [27]:
gdp_data

,Month,GDP
0,1997-01-01,61.9985
1,1997-02-01,62.5399
2,1997-03-01,62.5939
3,1997-04-01,63.2687
4,1997-05-01,62.6762
...,...,...
337,2025-02-01,102.3612
338,2025-03-01,102.7229
339,2025-04-01,102.6167
340,2025-05-01,102.5286


## CPIH

In [29]:
cpih = pd.read_csv("../../data/cpih.csv", skiprows=189)
cpih.rename(columns={'2025 Q2': 'year_month', '4.1': 'CPIH'}, inplace=True)
cpih['year_month'] = pd.to_datetime(cpih['year_month'], format='%Y %b')


In [30]:
cpih

,year_month,CPIH
0,1989-01-01,5.7
1,1989-02-01,5.8
2,1989-03-01,5.9
3,1989-04-01,5.6
4,1989-05-01,5.9
...,...,...
433,2025-02-01,3.7
434,2025-03-01,3.4
435,2025-04-01,4.1
436,2025-05-01,4.0


,Financial year,Local authority: district or unitary,Local authority code,Value,mid year pop estimate month,annual year,Quarters,dwelling_stock_year,year_month,Month,GDP,CPIH
0,1967/68,NaN,NaN,NaN,1967,1967,1967 Q3,1967,1967-07-01,NaT,NaN,NaN
1,1967/68,NaN,NaN,NaN,1967,1967,1967 Q3,1967,1967-08-01,NaT,NaN,NaN
2,1967/68,NaN,NaN,NaN,1967,1967,1967 Q3,1967,1967-09-01,NaT,NaN,NaN
3,1967/68,NaN,NaN,NaN,1967,1967,1967 Q4,1967,1967-10-01,NaT,NaN,NaN
4,1967/68,NaN,NaN,NaN,1967,1967,1967 Q4,1967,1967-11-01,NaT,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
100825,2026/27,NaN,NaN,NaN,2026,2026,2026 Q3,2026,2026-08-01,NaT,NaN,NaN
100826,2026/27,NaN,NaN,NaN,2026,2026,2026 Q3,2026,2026-09-01,NaT,NaN,NaN
100827,2026/27,NaN,NaN,NaN,2026,2026,2026 Q4,2026,2026-10-01,NaT,NaN,NaN
100828,2026/27,NaN,NaN,NaN,2026,2026,2026 Q4,2026,2026-11-01,NaT,NaN,NaN


## Combine data

In [33]:
# dwelling
additional_data = pd.merge(
    gdf[['LAD25CD', 'LAD25NM']],
    dwelling_stock,
    how='left',
    left_on='LAD25CD',
    right_on='NewONScode'
).drop(columns=['NewONScode', 'Area', 'year'])

# population
additional_data = pd.merge(
    additional_data,
    pop_by_la,
    how='left',
    left_on=['LAD25CD', 'year_month'],
    right_on=['LAD25CD', 'year_month']
).drop(columns=['year', 'mid year pop estimate month', 'local authority: district / unitary (as of April 2023)'])

# earnings
additional_data = pd.merge(
    additional_data,
    ashe_la,
    how='left',
    left_on=['LAD25CD', 'year_month'],
    right_on=['LAD25CD', 'year_month']
)

# base rate
additional_data = pd.merge(
    additional_data,
    base_rate,
    how='left',
    left_on='year_month',
    right_on='Date' 
).drop(columns=['Date'])

additional_data = additional_data.query("year_month >= '2007-04-01'")

# claimant count
additional_data = pd.merge(
    additional_data,
    claimant_count,
    how='left',
    left_on=['LAD25CD', 'year_month'],
    right_on=['mnemonic', 'year_month']
).drop(columns=['mnemonic', 'Description_ashe', 'local authority: district / unitary (as of April 2023)'])

# planning applications
additional_data = pd.merge(
    additional_data,
    planning_apps_granted_prop,
    how='left',
    left_on=['LAD25CD', 'year_month'],
    right_on=['LAD25CD', 'year_month']
)

# ORR data
additional_data = pd.merge(
    additional_data,
    orr_data,
    how='left',
    left_on=['LAD25CD', 'year_month'],
    right_on=['LAD25CD', 'year_month']
)

# GDP data
additional_data = pd.merge(
    additional_data,
    gdp_data,
    how='left',
    left_on='year_month',
    right_on='Month'
).drop(columns=['Month'])

# CPIH data
additional_data = pd.merge(
    additional_data,
    cpih,
    how='left',
    left_on='year_month',
    right_on='year_month'
)

additional_data

,LAD25CD,LAD25NM_x,dwelling_stock,year_month,population,ashe_weekly,base_rate,claimant_count_prop,LAD25NM_y,Prop_granted,Financial year,LAD25NM,Value,GDP,CPIH
0,E06000001,Hartlepool,41113.8,2007-04-01,90781.0,316.6,5.25,4.4,Hartlepool,0.954774,2007/08,Hartlepool,422020.0,83.4645,2.7
1,E06000001,Hartlepool,41113.8,2007-05-01,90781.0,316.6,5.50,4.3,Hartlepool,0.954774,2007/08,Hartlepool,422020.0,83.9159,2.5
2,E06000001,Hartlepool,41113.8,2007-06-01,90781.0,316.6,5.50,4.1,Hartlepool,0.954774,2007/08,Hartlepool,422020.0,84.0324,2.5
3,E06000001,Hartlepool,41113.8,2007-07-01,90969.0,316.6,5.75,4.1,Hartlepool,0.956731,2007/08,Hartlepool,422020.0,83.9629,2.0
4,E06000001,Hartlepool,41113.8,2007-08-01,90969.0,316.6,5.75,4,Hartlepool,0.956731,2007/08,Hartlepool,422020.0,84.4972,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63931,E09000033,Westminster,132895.0,2024-11-01,209996.0,947.4,4.75,4.2,Westminster,0.881928,NaN,NaN,NaN,101.4706,3.5
63932,E09000033,Westminster,132895.0,2024-12-01,209996.0,947.4,4.75,4.2,Westminster,0.881928,NaN,NaN,NaN,101.9167,3.5
63933,E09000033,Westminster,132895.0,2025-01-01,209996.0,947.4,4.75,4.2,Westminster,0.877023,NaN,NaN,NaN,101.8995,3.9
63934,E09000033,Westminster,132895.0,2025-02-01,209996.0,947.4,4.50,4.4,Westminster,0.877023,NaN,NaN,NaN,102.3612,3.7


In [98]:
additional_data

,LAD25CD,LAD25NM,dwelling_stock,year_month,population,Description_ashe,ashe_weekly,base_rate
72,E06000001,Hartlepool,41113.8,2007-04-01,90781.0,Hartlepool UA,316.6,5.25
73,E06000001,Hartlepool,41113.8,2007-05-01,90781.0,Hartlepool UA,316.6,5.50
74,E06000001,Hartlepool,41113.8,2007-06-01,90781.0,Hartlepool UA,316.6,5.50
75,E06000001,Hartlepool,41113.8,2007-07-01,90969.0,Hartlepool UA,316.6,5.75
76,E06000001,Hartlepool,41113.8,2007-08-01,90969.0,Hartlepool UA,316.6,5.75
...,...,...,...,...,...,...,...,...
85243,E09000033,Westminster,132895.0,2024-11-01,209996.0,Westminster,947.4,4.75
85244,E09000033,Westminster,132895.0,2024-12-01,209996.0,Westminster,947.4,4.75
85245,E09000033,Westminster,132895.0,2025-01-01,209996.0,Westminster,947.4,4.75
85246,E09000033,Westminster,132895.0,2025-02-01,209996.0,Westminster,947.4,4.50


63936 rows expected

In [101]:
planning_apps_granted_prop.query("LAD25CD == 'E06000001'")

,LAD25NM,LAD25CD,Prop_granted,year_month
426,Hartlepool,E06000001,0.924658,2008-01-01
427,Hartlepool,E06000001,0.924658,2008-02-01
428,Hartlepool,E06000001,0.924658,2008-03-01
1572,Hartlepool,E06000001,0.945946,2008-04-01
1573,Hartlepool,E06000001,0.945946,2008-05-01
...,...,...,...,...
68659,Hartlepool,E06000001,0.964912,2025-02-01
68660,Hartlepool,E06000001,0.964912,2025-03-01
69588,Hartlepool,E06000001,0.918033,2025-04-01
69589,Hartlepool,E06000001,0.918033,2025-05-01


In [53]:
additional_data = pd.merge(
    gdf[['LAD25CD', 'LAD25NM']],
    ashe_la,
    how='left',
    left_on='LAD25CD',
    right_on='LAD25CD'
)

In [54]:
ashe_la

,Description_ashe,LAD25CD,ashe_weekly,year_month
0,Adur,E07000223,327.2,2007-04-01
1,Adur,E07000223,327.2,2007-05-01
2,Adur,E07000223,327.2,2007-06-01
3,Adur,E07000223,327.2,2007-07-01
4,Adur,E07000223,327.2,2007-08-01
...,...,...,...,...
63931,York UA,E06000014,619.2,2024-11-01
63932,York UA,E06000014,619.2,2024-12-01
63933,York UA,E06000014,619.2,2025-01-01
63934,York UA,E06000014,619.2,2025-02-01


In [114]:
additional_data.to_excel("test_additional_data_join.xlsx", sheet_name='Sheet1', index=False, engine = 'openpyxl')

In [85]:
gdf[['LAD25CD', 'LAD25NM']].to_excel("gdf_la_codes.xlsx", sheet_name='Sheet1', index=False, engine = 'openpyxl')